# Mission 4: The Strategy Library — From Factors to Portfolios

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_04_strategy_library/notebook.ipynb)

**Learning objectives**
- Understand the economic intuition behind canonical quant strategies (momentum, value, quality, size, risk-based)
- Run a benchmark tournament across all built-in strategies using a single function call
- Diagnose *why* strategies fail: crowding, transaction costs, macro regimes
- Build your own composite signal using IC-weighted combination
- (Challenge) Benchmark your Mission 3 submission against the canonical strategies

## Part 0: Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi-lab>=0.1.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from convexpi.lab import (
    SyntheticMarket,
    Backtest,
    STRATEGIES,
    compare,
    # Individual strategy classes
    CrossSectionalMomentum,
    ShortTermReversal,
    TimeSeriesMomentum,
    ValueTilt,
    QualityTilt,
    BettingAgainstBeta,
    SizePremium,
    FamaFrench3,
    MultiFactorRank,
    ICWeightedComposite,
    InverseVolatilityWeight,
    MinimumVarianceScreen,
    TrendFilter,
    DualMomentum,
    MacroCyclical,
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready. Strategy count:', len(STRATEGIES))

---
## Part 1: The Market

We use the same **synthetic market** as Missions 1–3.  The market has 200 stocks and 1,000 trading days.  The first 750 days are the training window (IS); the last 250 are the hidden holdout (OOS).

**Key features available to strategies:**

| Feature | Description | Used by |
|---|---|---|
| `mom_12m` | 12-month momentum (skip last month) | Momentum, FF3, Dual |
| `mom_3m` | 3-month momentum | Momentum |
| `mom_1m` | 1-month momentum (reversal signal) | Reversal |
| `reversal_1w` | 1-week reversal | Reversal |
| `vol_1m` | 1-month realized volatility | BAB, InvVol, MinVar |
| `val_bm` | Value: book-to-market proxy | Value, FF3 |
| `qual_roe` | Quality: return-on-equity proxy | Quality, FF3 |
| `size_cap` | Size: log market cap | Size, FF3 |

In [ ]:
market = SyntheticMarket(n_stocks=200, n_days=1000, seed=42)

print(f"Stocks  : {market.n_stocks}")
print(f"Days    : {market.n_days}")
print(f"Train   : days 0..{market.train_end - 1}  ({market.train_end} days)")
print(f"Holdout : days {market.train_end}..{market.n_days - 1}  ({market.n_days - market.train_end} days)")
print(f"Features: {market.FEATURE_NAMES}")

---
## Part 2: The Strategy Zoo

The strategy library contains **19 canonical quant strategies** organized into five families:

```
Momentum    — Cross-sectional, time-series, short-term reversal, dual momentum
Value       — Book-to-market tilt (long-short and long-only)
Quality     — ROE tilt, Betting-Against-Beta
Size        — Small-cap premium
Multi       — Fama-French 3, Rank-sum composite, IC-weighted composite
Risk-based  — Inverse volatility, minimum variance screen
Conditional — Trend filter, macro regime
```

All strategies share the **same interface**: `on_day(day, features, prices, portfolio) → weight_array`.  
The `Backtest` engine calls this once per day and handles rebalancing, turnover, and transaction costs.

Every strategy in `STRATEGIES` is pre-instantiated and ready to run:

In [ ]:
print(f"{'Strategy key':<25} {'Class'}")
print('-' * 60)
for key, strat in STRATEGIES.items():
    print(f"{key:<25} {type(strat).__name__}")

---
## Part 3: The Tournament

`compare()` runs every strategy through the same backtester and returns a single DataFrame with seven performance metrics.

This is the **horse race** view: raw performance before any attempt to understand *why* each strategy works.

In [ ]:
results = compare(
    STRATEGIES,
    market=market,
    split='train',      # IS window only — OOS is reserved
    warmup_days=63,     # ~3 months for lookbacks to populate
    tc_bps=10,          # 10 bps per-side transaction cost
)

results.sort_values('sharpe', ascending=False)

In [ ]:
# Visual: Sharpe bar chart
fig, ax = plt.subplots(figsize=(12, 5))
sorted_r = results.sort_values('sharpe', ascending=True)
colors = ['#2ecc71' if s > 0 else '#e74c3c' for s in sorted_r['sharpe']]
ax.barh(sorted_r.index, sorted_r['sharpe'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sharpe Ratio (IS, 10 bps TC)')
ax.set_title('Strategy Tournament — In-Sample Sharpe')
plt.tight_layout()
plt.show()

**Discussion questions:**

1. Which family of strategies dominates in this market?  Why might that be?
2. Some strategies have high return but low Sharpe — what does that imply about risk?
3. The equal-weight benchmark is included in the registry.  How do the "smart" strategies compare to it?

> **Hint:** Check the `annual_turnover` column.  A strategy with 40× annual turnover at 10 bps eats 4% per year in costs before any alpha.

---
## Part 4: Anatomy of a Strategy

Let's open up three strategies and see how each one translates a feature signal into a weight vector.

### 4a. Cross-Sectional Momentum (Jegadeesh & Titman, 1993)

Buy the past-year winners, short the past-year losers, skip the most recent month.

**Signal:** `mom_12m`  
**Weight construction:** rank → top quintile long, bottom quintile short, $|w|$ normalized to 1  
**Known failure modes:** crashes after sharp market rallies; decays at high transaction costs

In [ ]:
bt = Backtest(warmup_days=63, tc_bps=10)
mom = CrossSectionalMomentum()
r_mom = bt.run(mom, market.prices('train'), market.features('train'))
r_mom.tearsheet()

In [ ]:
# Inspect one day of weights to see the portfolio structure
prices = market.prices('train')
features = market.features('train')
features_day300 = {k: v[300] for k, v in features.items()}
w = mom.on_day(300, features_day300, prices[300], np.zeros(market.n_stocks))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Weight distribution
axes[0].hist(w[w != 0], bins=30, color='steelblue', alpha=0.8)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Weight Distribution (day 300)')
axes[0].set_xlabel('Portfolio weight')

# Signal vs weight scatter
sig = features_day300['mom_12m']
axes[1].scatter(sig, w, alpha=0.4, s=10)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('mom_12m signal')
axes[1].set_ylabel('Portfolio weight')
axes[1].set_title('Signal vs Weight')

plt.tight_layout()
plt.show()

print(f"Long positions : {(w > 0).sum()} stocks")
print(f"Short positions: {(w < 0).sum()} stocks")
print(f"Net exposure   : {w.sum():.4f}  (should be ~0)")

### 4b. Betting Against Beta (Frazzini & Pedersen, 2014)

Low-volatility stocks outperform on a risk-adjusted basis — the **low-risk anomaly**.  High-beta assets trade at a premium because leverage-constrained investors (mutual funds, insurance) are forced to overweight them.

**Signal:** `vol_1m` (lower vol → higher allocation)  
**Weight construction:** inverse of volatility rank  
**Economic intuition:** the CAPM predicts a linear risk-return tradeoff; in practice the Security Market Line is too flat

In [ ]:
bab = BettingAgainstBeta()
r_bab = bt.run(bab, market.prices('train'), market.features('train'))
r_bab.tearsheet()

### 4c. IC-Weighted Composite

Instead of pre-specifying factor weights, let the **Information Coefficient (IC)** decide.  The IC is the rank correlation between a signal on day $t$ and the actual return on day $t+1$.  Signals with recent high IC get more weight.

**Why it works:** factors go in and out of favor.  A static weight (Fama-French 3) is just a prior; the IC-weighted version adapts.  
**Risk:** IC estimation from short windows is noisy — the composite may overfit if the IC lookback is too short.

In [ ]:
ic = ICWeightedComposite(ic_window=60)
r_ic = bt.run(ic, market.prices('train'), market.features('train'))
r_ic.tearsheet()

---
## Part 5: Cumulative Returns Comparison

The tournament table flattens time.  Let's look at the equity curves of a curated set of strategies to understand *when* each approach earns its returns.

In [ ]:
focus = {
    'equal_weight'    : STRATEGIES['equal_weight'],
    'momentum_12_1'   : STRATEGIES['momentum_12_1'],
    'value_bm'        : STRATEGIES['value_bm'],
    'quality_roe'     : STRATEGIES['quality_roe'],
    'betting_against_beta': STRATEGIES['betting_against_beta'],
    'ic_weighted'     : STRATEGIES['ic_weighted'],
    'fama_french_3'   : STRATEGIES['fama_french_3'],
}

fig, ax = plt.subplots(figsize=(14, 6))

for name, strategy in focus.items():
    r = bt.run(strategy, market.prices('train'), market.features('train'))
    cum = r.cumulative_returns
    ax.plot(cum, label=name, linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
ax.set_title('Cumulative Returns — IS Window (10 bps TC)')
ax.set_xlabel('Trading day')
ax.set_ylabel('Cumulative return')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

**Discussion questions:**

1. Do any strategies show strongly correlated equity curves?  What does that imply about diversification?
2. Identify a period where momentum and value diverge.  What market regime might explain this?
3. Does the IC-weighted composite smooth out the worst drawdowns of its component strategies?

---
## Part 6: The Transaction Cost Tax

High-turnover strategies look great at 0 bps but collapse under realistic costs.  Let's quantify the breakeven spread for each strategy.

In [ ]:
tc_levels = [0, 5, 10, 20, 30]
focus_keys = ['equal_weight', 'momentum_12_1', 'momentum_3m', 'value_bm',
               'quality_roe', 'ic_weighted', 'inv_vol']

rows = []
for tc in tc_levels:
    bt_tc = Backtest(warmup_days=63, tc_bps=tc)
    for key in focus_keys:
        r = bt_tc.run(STRATEGIES[key], market.prices('train'), market.features('train'))
        rows.append({'strategy': key, 'tc_bps': tc, 'sharpe': r.sharpe})

tc_df = pd.DataFrame(rows)
tc_pivot = tc_df.pivot(index='strategy', columns='tc_bps', values='sharpe')
tc_pivot.columns = [f'{c} bps' for c in tc_pivot.columns]
tc_pivot.sort_values('0 bps', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for key in focus_keys:
    sub = tc_df[tc_df['strategy'] == key]
    ax.plot(sub['tc_bps'], sub['sharpe'], marker='o', label=key)

ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('Transaction cost (bps per side)')
ax.set_ylabel('Sharpe ratio')
ax.set_title('TC Sensitivity — IS Sharpe vs. One-Way Spread')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Key insight:** A strategy that turns over 30× per year loses **30 × 2 × TC** in annual costs.  At 10 bps two-way, that's **6% per year** before any alpha.  Check the `annual_turnover` column in the tournament results — that's the tax multiplier.

---
## Part 7: Build Your Own Composite

Rather than using a pre-built strategy, you can compose your own by instantiating strategy classes with custom parameters and combining their signals.

The `MultiFactorRank` strategy takes any list of signal keys and rank-sums them:

In [ ]:
# Custom composite: momentum + quality + value
my_composite = MultiFactorRank(
    signals=['mom_12m', 'qual_roe', 'val_bm'],
    long_only=False,
)

r_custom = bt.run(my_composite, market.prices('train'), market.features('train'))
r_custom.tearsheet()

In [ ]:
# Your turn — try your own combination
# Suggestions:
#   - Remove a signal and see if Sharpe improves (less noise)
#   - Add 'size_cap' (contrarian: negative weight on large caps)
#   - Try long_only=True and see how it changes the risk profile

my_strategy = MultiFactorRank(
    signals=['mom_12m', 'qual_roe'],  # EDIT THIS
    long_only=False,
)

r_mine = bt.run(my_strategy, market.prices('train'), market.features('train'))
r_mine.tearsheet()

---
## Part 8: IS vs OOS — The Reality Check

Everything above was **in-sample**.  We know from Mission 1 that IS performance is a biased estimate of OOS performance.  Let's run the top strategies on the **holdout window** and see what survives.

> **Rule:** You look at OOS *once*, at the end.  If you use OOS to select a strategy, it becomes IS.

In [ ]:
# Pick the top 5 IS performers
top5_keys = results.sort_values('sharpe', ascending=False).head(5).index.tolist()
top5_strategies = {k: STRATEGIES[k] for k in top5_keys}

print("Top 5 IS strategies:", top5_keys)

is_results  = compare(top5_strategies, market=market, split='train',   warmup_days=63, tc_bps=10)
oos_results = compare(top5_strategies, market=market, split='holdout', warmup_days=63, tc_bps=10)

comparison = pd.DataFrame({
    'IS Sharpe' : is_results['sharpe'],
    'OOS Sharpe': oos_results['sharpe'],
    'IS Return' : is_results['annual_return'],
    'OOS Return': oos_results['annual_return'],
    'Turnover'  : is_results['annual_turnover'],
}).sort_values('IS Sharpe', ascending=False)

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison))
width = 0.35

ax.bar(x - width/2, comparison['IS Sharpe'],  width, label='IS Sharpe',  color='steelblue', alpha=0.8)
ax.bar(x + width/2, comparison['OOS Sharpe'], width, label='OOS Sharpe', color='coral',     alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=20, ha='right')
ax.set_ylabel('Sharpe ratio')
ax.set_title('Top-5 IS Strategies: IS vs OOS Sharpe')
ax.legend()
plt.tight_layout()
plt.show()

**Expected finding:** OOS Sharpe is almost always lower than IS Sharpe.  The gap measures overfitting — even on canonical strategies that nobody explicitly fit to this data.  This is the **dimensionality problem**: with 200 stocks and 19 strategies, you have enough degrees of freedom to accidentally select lucky strategies.

---
## Part 9: Challenges

Complete any **two** of the following.

### Challenge A: The Low-Turnover Portfolio

Build a strategy with Sharpe > 0.3 IS *and* annual turnover < 5×.  
Hint: `InverseVolatilityWeight` and `MinimumVarianceScreen` have very low natural turnover.

In [ ]:
# Challenge A — your solution here
from convexpi.lab import Strategy
import numpy as np

class LowTurnoverStrategy(Strategy):
    def on_day(self, day, features, prices, portfolio):
        # Your implementation
        n = len(prices)
        return np.zeros(n)

bt_a = Backtest(warmup_days=63, tc_bps=10)
r_a = bt_a.run(LowTurnoverStrategy(), market.prices('train'), market.features('train'))
r_a.tearsheet()
print(f"Turnover: {r_a.turnover_annual:.1f}×  (target < 5×)")
print(f"Sharpe  : {r_a.sharpe:.3f}   (target > 0.3)")

### Challenge B: The Regime Filter

Momentum strategies are known to crash after volatile downturns.  Use `TrendFilter` or `DualMomentum` to switch momentum off during drawdowns.  Does this improve the Sharpe-to-max-drawdown ratio (Calmar)?

In [ ]:
# Challenge B — your solution here
raw_mom     = CrossSectionalMomentum()
filtered_mom = TrendFilter(inner=CrossSectionalMomentum(), flat_on_down=True)

r_raw      = bt.run(raw_mom,     market.prices('train'), market.features('train'))
r_filtered = bt.run(filtered_mom, market.prices('train'), market.features('train'))

print("Raw momentum")
r_raw.tearsheet()
print("\nTrend-filtered momentum")
r_filtered.tearsheet()

### Challenge C: Beat the Registry

Design a strategy that achieves higher OOS Sharpe than all 19 built-in strategies.  You may not look at the holdout window while developing your strategy.  Document your reasoning.

In [ ]:
# Challenge C — your solution here
class MyBestStrategy(Strategy):
    def on_day(self, day, features, prices, portfolio):
        n = len(prices)
        return np.zeros(n)

# IS development
r_is = bt.run(MyBestStrategy(), market.prices('train'), market.features('train'))
r_is.tearsheet()

# Final OOS check (do this ONCE, at the end)
bt_oos = Backtest(warmup_days=63, tc_bps=10)
r_oos = bt_oos.run(MyBestStrategy(), market.prices('holdout'), market.features('holdout'))
print(f"\nOOS Sharpe: {r_oos.sharpe:.3f}")
best_oos = oos_results['sharpe'].max()
print(f"Best registry OOS Sharpe: {best_oos:.3f}")
print(f"Beat registry: {r_oos.sharpe > best_oos}")

---
## Wrap-up

**What you covered:**

| Concept | Where |
|---|---|
| Strategy taxonomy (momentum, value, quality, size, risk) | Part 2 |
| Running a strategy tournament with `compare()` | Part 3 |
| Portfolio anatomy: weight distribution & signal-weight relationship | Part 4 |
| Equity curve divergence across regimes | Part 5 |
| Transaction cost drag as a function of turnover | Part 6 |
| Building custom composites with `MultiFactorRank` | Part 7 |
| IS vs OOS validation — even for canonical strategies | Part 8 |

**Key takeaways:**

1. **No single strategy dominates all regimes.** The IS winner is rarely the OOS winner.
2. **Transaction costs are multiplicative with turnover.** A 30× turnover strategy needs 30× more gross alpha to break even.
3. **IC-weighting adapts but can overfit.** A 60-day IC window is too short to be reliable; use it as one input, not the whole model.
4. **Regime filtering helps, but you must define the regime ex ante.** Post-hoc filtering is data snooping.

---

**Ready for Mission 5?**  
Mission 5 applies the real-data Lab mode — pulling Ken French factor data and FRED macro series to run these same strategies on actual market history.